# PREP_01 - Train /  Test / Validation Split

## TFM - Skin Lesion Classification (ISIC 2024 / SLICE-3D)

Con el objetivo de obtener los datasets que utilizaremos para entrenar nuestros modelos de clasificación en este notebook vamos a trabajar a partir de los datasets resultantes de los scripts preprocess_data.py y final_preprocess_data.py

En concreto trabajaremos con el artefacto ya generado:

* **Final preprocessed metadata** (`final_preprocessed_from_raw_<timestamp>.parquet`) — Es el resultado de EDA 01 y 02 y contiene todos los datos que necesitamos para hacer el split.



## Imports

In [1]:
import pandas as pd
import numpy as np


from sklearn.model_selection import StratifiedGroupKFold


from skin_lesion_ai.utils.data_utils import (
    load_metadata_parquet,
    save_metadata_parquet,
)

from skin_lesion_ai.visualisation.eda_plots import (
    set_eda_style,
    clean_axis_labels,
)

# Set the EDA style
set_eda_style()

## Cargar Dataset

Cargamos el dataset utilizando la función creada para tal efecto. 

In [2]:
# Load the preprocessed metadata
df_preprocessed = load_metadata_parquet(
    stage="processed",
    filename="final_preprocessed_from_raw",
    timestamp_flag=True,
)

print(f"Final preprocessed metadata: {df_preprocessed.shape}")
df_preprocessed.head()

Final preprocessed metadata: (381280, 17)


,isic_id,patient_id,diagnostic_group,target_biopsy,target_malignant,sex,sex_male,age_approx,anatom_site_general,anatom_site_general_code,anatom_site__anterior_torso,anatom_site__head_neck,anatom_site__lower_extremity,anatom_site__posterior_torso,anatom_site__upper_extremity,clin_size_long_diam_mm,clin_size_long_diam_mm_log1p
0,ISIC_0015670,IP_1235828,benign_non_biopsied,0,<NA>,male,1,60.0,lower extremity,4,0,0,1,0,0,3.04,1.396245
1,ISIC_0015845,IP_8170065,benign_non_biopsied,0,<NA>,male,1,60.0,head/neck,5,0,1,0,0,0,1.10,0.741937
2,ISIC_0015864,IP_6724798,benign_non_biopsied,0,<NA>,male,1,60.0,posterior torso,2,0,0,0,1,0,3.40,1.481605
3,ISIC_0015902,IP_4111386,benign_non_biopsied,0,<NA>,male,1,65.0,anterior torso,1,1,0,0,0,0,3.22,1.439835
4,ISIC_0024200,IP_8313778,benign_non_biopsied,0,<NA>,male,1,55.0,anterior torso,1,1,0,0,0,0,2.73,1.316408


In [3]:
# Comprobamos la información del DataFrame preprocesado
df_preprocessed.info()

<class 'pandas.DataFrame'>
RangeIndex: 381280 entries, 0 to 381279
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   isic_id                       381280 non-null  str    
 1   patient_id                    381280 non-null  str    
 2   diagnostic_group              381280 non-null  string 
 3   target_biopsy                 381280 non-null  int8   
 4   target_malignant              1013 non-null    Int8   
 5   sex                           381280 non-null  str    
 6   sex_male                      381280 non-null  int8   
 7   age_approx                    381280 non-null  float64
 8   anatom_site_general           381280 non-null  str    
 9   anatom_site_general_code      381280 non-null  int8   
 10  anatom_site__anterior_torso   381280 non-null  int8   
 11  anatom_site__head_neck        381280 non-null  int8   
 12  anatom_site__lower_extremity  381280 non-null  int8   


## Using splitter to split df stratified and by groupal id

Para hacer la partición del dataset en los correspondientes train / test / validation vamos a utilizar como splitter la función StratifiedGroupKFold ya que nos va a permitir aplicar a la partición todas las condiciones que nos interesa para operar.

Con StratifiedGroupKFold podemos hacer que la partición se haga agrupando las lesiones por paciente (grouping), que trate de respetar la estructura de 'target_biopsy' (stratification) y haga shuffle. La limitación que tenemos con StratifiedGroupKFold es que no en todos los casos obtendremos particiones del tamaño que deseamos, ya que está diseñada para cross-validation y trabaja a partir del número de splits que pretendemos hacer.


Para más información podemos consultar las fuentes donde hemos aprendido:


Issue discusion on how/why use the function StratifiedGroupKFold to do the split: https://github.com/scikit-learn/scikit-learn/issues/12076

Detailed explanation on split: https://github.com/scikit-learn/scikit-learn/issues/9193

Aplied example: https://stackoverflow.com/questions/56872664/complex-dataset-split-stratifiedgroupshufflesplit


In [4]:
# Split patients into train and test_val sets
# El test_size esta relacionado inversamente con el numero de folds
# Pero el n_folds debe ser entero

random_state = 42
test_size = 0.2
desired = 1.0 / test_size
n_folds = int(np.round(desired)) 
# De manera alternativa: c=np.ceil(desired), f=np.floor(desired), c if c/desired < desired /f else f

# Creamos objeto splitter
splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=random_state)


# Split manteniendo stratificacion de 'target_biopsy' y agrupando por 'patient_id'
split = splitter.split(X=df_preprocessed,y=df_preprocessed['target_biopsy'], groups=df_preprocessed['patient_id'])
train_inds, test_val_inds = next(split)

train_df = df_preprocessed.iloc[train_inds]
test_val_df = df_preprocessed.iloc[test_val_inds]

In [ ]:
# Debemos hacer split de nuevo para separar test y validation

val_size = 0.5
desired_val = 1.0 / val_size
n_folds_val = int(np.round(desired_val)) 

# Creamos obj splitter para el split que haremos
splitter_val = StratifiedGroupKFold(n_splits=n_folds_val, shuffle=True, random_state=random_state)

# Creamos el obj split manteniendo stratificacion de 'target_biopsy' y agrupando por 'patient_id'
split_val = splitter_val.split(X=test_val_df,y=test_val_df['target_biopsy'], groups=test_val_df['patient_id'])

# Aplicamos el split 1 iteracion
test_inds, val_inds = next(split_val)

test_df = test_val_df.iloc[test_inds]
val_df = test_val_df.iloc[val_inds]

## Validacion de los splits

El chequeo más inmediato que podemos hacer es comprobar la longitud de los sets.

In [6]:
# Check train and test sets length
print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Val set: {len(val_df)} samples")

# Percentage of the original dataset
print(f"Train set: {len(train_df)*100/len(df_preprocessed):.2f}% samples")
print(f"Test set: {len(test_df)*100/len(df_preprocessed):.2f}% samples")
print(f"Val set: {len(val_df)*100/len(df_preprocessed):.2f}% samples")

Train set: 305024 samples
Test set: 38125 samples
Val set: 38131 samples
Train set: 80.00% samples
Test set: 10.00% samples
Val set: 10.00% samples


Vamos a comprobar que no haya ningún paciente en ambos splits a la vez.

In [ ]:
# Patient check overlaps

df_overlap1 = pd.merge(train_df, test_df,how="inner", on=["patient_id","patient_id"])
print(f"Train ∩ Test overlap: {len(df_overlap1)} patients")

df_overlap2 = pd.merge(val_df, test_df,how="inner", on=["patient_id","patient_id"])
print(f"Validation ∩ Test overlap: {len(df_overlap2)} patients")

df_overlap3 = pd.merge(val_df, train_df,how="inner", on=["patient_id","patient_id"])
print(f"Validation ∩ Train overlap: {len(df_overlap3)} patients")

if (len(df_overlap1)) + len(df_overlap2) + len(df_overlap3) == 0:
    print("✅ No patient overlap")
else:
    print("❌ Patient overlap detected")


Train ∩ Test overlap: 0 patients
Validation ∩ Test overlap: 0 patients
Validation ∩ Train overlap: 0 patients
✅ No patient overlap


Verificamos que la partición la hemos hecho stratificada y, por lo tanto, mantenemos las proporciones en el split.

In [8]:
# checkear stratificacion de nuestro split

preprocessed_y = df_preprocessed['target_biopsy'].value_counts()
train_y = train_df['target_biopsy'].value_counts()
test_y = test_df['target_biopsy'].value_counts()
val_y = val_df['target_biopsy'].value_counts()

print(f"Preprocessed_df target_biopsy samples count:\n {preprocessed_y[0]} with value '0' \n {preprocessed_y[1]} with value '1' \n")
print(f"As percentatge:\n {100*preprocessed_y[0]/len(df_preprocessed):.2f}% with value '0' \n {100*preprocessed_y[1]/len(df_preprocessed):.2f}% with value '1' \n" )

print("\nAfter our split we have: \n")

print(f"Train_df target_biopsy samples count:\n {train_y[0]} with value '0' \n {train_y[1]} with value '1' \n")
print(f"As percentatge:\n {100*train_y[0]/len(train_df):.2f}% with value '0' \n {100*train_y[1]/len(train_df):.2f}% with value '1' \n" )

print(f"Test_df target_biopsy samples count:\n {test_y[0]} with value '0' \n {test_y[1]} with value '1' \n")
print(f"As percentatge:\n {100*test_y[0]/len(test_df):.2f}% with value '0' \n {100*test_y[1]/len(test_df):.2f}% with value '1' \n" )

print(f"Val_df target_biopsy samples count:\n {val_y[0]} with value '0' \n {val_y[1]} with value '1' \n")
print(f"As percentatge:\n {100*val_y[0]/len(val_df):.2f}% with value '0' \n {100*val_y[1]/len(val_df):.2f}% with value '1' \n" )


Preprocessed_df target_biopsy samples count:
 380267 with value '0' 
 1013 with value '1' 

As percentatge:
 99.73% with value '0' 
 0.27% with value '1' 


After our split we have: 

Train_df target_biopsy samples count:
 304212 with value '0' 
 812 with value '1' 

As percentatge:
 99.73% with value '0' 
 0.27% with value '1' 

Test_df target_biopsy samples count:
 38025 with value '0' 
 100 with value '1' 

As percentatge:
 99.74% with value '0' 
 0.26% with value '1' 

Val_df target_biopsy samples count:
 38030 with value '0' 
 101 with value '1' 

As percentatge:
 99.74% with value '0' 
 0.26% with value '1' 



In [9]:
# Checkear stratification para cada split en % hasta 1 decimal

if np.round(100*preprocessed_y[0]/len(df_preprocessed), 1) == np.round(100*train_y[0]/len(train_df), 1):
    print("✅ Stratification maintained for value '0' in train set")
else:
    print("❌ Stratification not maintained for value '0' in train set")

if np.round(100*preprocessed_y[1]/len(df_preprocessed), 1) == np.round(100*train_y[1]/len(train_df), 1):
    print("✅ Stratification maintained for value '1' in train set")
else:
    print("❌ Stratification not maintained for value '1' in train set")

# Check stratification for test set
if np.round(100*preprocessed_y[0]/len(df_preprocessed), 1) == np.round(100*test_y[0]/len(test_df), 1):
    print("✅ Stratification maintained for value '0' in test set")
else:
    print("❌ Stratification not maintained for value '0' in test set")

if np.round(100*preprocessed_y[1]/len(df_preprocessed), 1) == np.round(100*test_y[1]/len(test_df), 1):
    print("✅ Stratification maintained for value '1' in test set")
else:
    print("❌ Stratification not maintained for value '1' in test set")

# Check stratification for validation set
if np.round(100*preprocessed_y[0]/len(df_preprocessed), 1) == np.round(100*val_y[0]/len(val_df), 1):
    print("✅ Stratification maintained for value '0' in validation set")
else:
    print("❌ Stratification not maintained for value '0' in validation set")

if np.round(100*preprocessed_y[1]/len(df_preprocessed), 1) == np.round(100*val_y[1]/len(val_df), 1):
    print("✅ Stratification maintained for value '1' in validation set")
else:
    print("❌ Stratification not maintained for value '1' in validation set")

✅ Stratification maintained for value '0' in train set
✅ Stratification maintained for value '1' in train set
✅ Stratification maintained for value '0' in test set
✅ Stratification maintained for value '1' in test set
✅ Stratification maintained for value '0' in validation set
✅ Stratification maintained for value '1' in validation set


Comprobamos la coherencia del número de lesiones

In [10]:
# numero de lesiones en cada split y porcentaje del original (preprocessed)
print(f"Preprocessed set:\n {df_preprocessed['isic_id'].nunique()} lesions\n")

print(f"Train set:\n {train_df['isic_id'].nunique()} lesions")
print(f" {100*train_df['isic_id'].nunique()/df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n")

print(f"Test set:\n {test_df['isic_id'].nunique()} lesions")
print(f" {100*test_df['isic_id'].nunique()/df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n")

print(f"Validation set:\n {val_df['isic_id'].nunique()} lesions")
print(f" {100*val_df['isic_id'].nunique()/df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n")

# se cumple igualdad de train + test + val = preprocessed
print(f"Total lesions in splits: {train_df['isic_id'].nunique() + test_df['isic_id'].nunique() + val_df['isic_id'].nunique()} lesions")
print(f"Preprocessed lesions: {df_preprocessed['isic_id'].nunique()} lesions")

if train_df['isic_id'].nunique() + test_df['isic_id'].nunique() + val_df['isic_id'].nunique() == df_preprocessed['isic_id'].nunique():
    print("✅ Train + Test + Val = Preprocessed")
else:
    print("❌ Train + Test + Val != Preprocessed")

Preprocessed set:
 381280 lesions

Train set:
 305024 lesions
 80.0 % 
of preprocessed set

Test set:
 38125 lesions
 9.999213176668066 % 
of preprocessed set

Validation set:
 38131 lesions
 10.000786823331934 % 
of preprocessed set

Total lesions in splits: 381280 lesions
Preprocessed lesions: 381280 lesions
✅ Train + Test + Val = Preprocessed


Validamos que hay coherencia con el numero de pacientes

In [11]:
# numero de pacientes en cada split
print(f"Preprocessed set:\n {df_preprocessed['patient_id'].nunique()} patients\n")

print(f"Train set:\n {train_df['patient_id'].nunique()} patients")
print(f" {100*train_df['patient_id'].nunique()/df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n")

print(f"Test set:\n {test_df['patient_id'].nunique()} patients")
print(f" {100*test_df['patient_id'].nunique()/df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n")

print(f"Validation set:\n {val_df['patient_id'].nunique()} patients")
print(f" {100*val_df['patient_id'].nunique()/df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n")

# Se cumple igualdad de train + test + val = preprocessed
print(f"Total patients in splits: {train_df['patient_id'].nunique() + test_df['patient_id'].nunique() + val_df['patient_id'].nunique()} patients")
print(f"Preprocessed patients: {df_preprocessed['patient_id'].nunique()} patients")

if train_df['patient_id'].nunique() + test_df['patient_id'].nunique() + val_df['patient_id'].nunique() == df_preprocessed['patient_id'].nunique():
    print("✅ Train + Test + Val = Preprocessed")
else:
    print("❌ Train + Test + Val != Preprocessed")

Preprocessed set:
 977 patients

Train set:
 784 patients
 80.24564994882293 % 
of preprocessed set

Test set:
 99 patients
 10.133060388945752 % 
of preprocessed set

Validation set:
 94 patients
 9.62128966223132 % 
of preprocessed set

Total patients in splits: 977 patients
Preprocessed patients: 977 patients
✅ Train + Test + Val = Preprocessed


## Data save

Una vez hemos comprobado que el split lo hemos operado tal y como pretendíamos debemos guardar el resultado de nuestro proceso así como los índices subproducto del proceso para garantizar la reproducibilidad y revisión del proceso.


In [12]:
# Guardamos los splits resultantes


train_path = save_metadata_parquet(
    train_df,
    stage="processed",
    name="train",
    timestamp=True,
)



val_path = save_metadata_parquet(
    val_df,
    stage="processed",
    name="validation",
    timestamp=True,
)


test_path = save_metadata_parquet(
    test_df,
    stage="processed",
    name="test",
    timestamp=True,
)

print(f"Train saved to: {train_path}")
print(f"Validation saved to: {val_path}")
print(f"Test saved to: {test_path}")

# Guardamos los indices de los procesos de split
# debemos crear un DataFrame con los indices de cada split y guardarlo en un archivo parquet


split_indices_path = save_metadata_parquet(
    pd.DataFrame({"train_indices": pd.Series(train_inds), "test_val_indices": pd.Series(test_val_inds),
                   "test_indices": pd.Series(test_inds), "val_indices": pd.Series(val_inds)}),
    stage="interim",
    name="split_indices",
    timestamp=True,
)

print(f"Split indices saved to: {split_indices_path}")

Train saved to: C:\Users\Krop\Desktop\Master\Curso_Master\TFM\ProjecteTFM\my-image-classifier\data\processed\metadata\train_20260731_184144.parquet
Validation saved to: C:\Users\Krop\Desktop\Master\Curso_Master\TFM\ProjecteTFM\my-image-classifier\data\processed\metadata\validation_20260731_184144.parquet
Test saved to: C:\Users\Krop\Desktop\Master\Curso_Master\TFM\ProjecteTFM\my-image-classifier\data\processed\metadata\test_20260731_184144.parquet
Split indices saved to: C:\Users\Krop\Desktop\Master\Curso_Master\TFM\ProjecteTFM\my-image-classifier\data\interim\metadata\split_indices_20260731_184144.parquet
